In [17]:
import os
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull, distance_matrix
from scipy.io import loadmat
from pathlib import Path


In [18]:
output_folder = "../results/output_monolayer/"
files = sorted(Path(output_folder).glob('*output*_cells.mat'))
files

[PosixPath('../results/output_monolayer/output00000000_cells.mat'),
 PosixPath('../results/output_monolayer/output00000001_cells.mat'),
 PosixPath('../results/output_monolayer/output00000002_cells.mat'),
 PosixPath('../results/output_monolayer/output00000003_cells.mat'),
 PosixPath('../results/output_monolayer/output00000004_cells.mat'),
 PosixPath('../results/output_monolayer/output00000005_cells.mat'),
 PosixPath('../results/output_monolayer/output00000006_cells.mat'),
 PosixPath('../results/output_monolayer/output00000007_cells.mat'),
 PosixPath('../results/output_monolayer/output00000008_cells.mat'),
 PosixPath('../results/output_monolayer/output00000009_cells.mat'),
 PosixPath('../results/output_monolayer/output00000010_cells.mat'),
 PosixPath('../results/output_monolayer/output00000011_cells.mat'),
 PosixPath('../results/output_monolayer/output00000012_cells.mat'),
 PosixPath('../results/output_monolayer/output00000013_cells.mat'),
 PosixPath('../results/output_monolayer/output00

In [19]:
# Loop through files
interval = 720
i=0
df_cell = pd.DataFrame()
results = []

for file in files:
    mat = loadmat(file)
    cell_data = mat['cells'][[0, 1, 2, 3]]
    df_mat = pd.DataFrame(cell_data.T, columns=['id', 'x', 'y', 'z'])
    df_mat['dt'] =i * interval
    df_cell = pd.concat([df_cell, df_mat], ignore_index=True)
    x = df_mat['x'].values
    y = df_mat['y'].values
    cell_count = len(x)
    if cell_count < 2:
        diameter = 0.0
    else:
        try:
            points = np.column_stack((x, y))
            hull = ConvexHull(points)
            hull_points = points[hull.vertices]
            distances = distance_matrix(hull_points, hull_points)
            diameter = distances.max()
        except:
            diameter = 0.0
    
    i += 1
    results.append({
    'timestep': i * interval,
    'cell_count': cell_count,
    'diameter': diameter
})
results

[{'timestep': 720, 'cell_count': 1, 'diameter': 0.0},
 {'timestep': 1440, 'cell_count': 1, 'diameter': 0.0},
 {'timestep': 2160, 'cell_count': 2, 'diameter': 0.0},
 {'timestep': 2880,
  'cell_count': 4,
  'diameter': np.float64(20.114048823097797)},
 {'timestep': 3600,
  'cell_count': 4,
  'diameter': np.float64(21.935495853260992)},
 {'timestep': 4320,
  'cell_count': 8,
  'diameter': np.float64(31.230939877443586)},
 {'timestep': 5040,
  'cell_count': 16,
  'diameter': np.float64(44.19268287982527)},
 {'timestep': 5760,
  'cell_count': 16,
  'diameter': np.float64(51.929798663516685)},
 {'timestep': 6480,
  'cell_count': 32,
  'diameter': np.float64(68.0448303326553)},
 {'timestep': 7200,
  'cell_count': 64,
  'diameter': np.float64(84.67615739844005)},
 {'timestep': 7920,
  'cell_count': 64,
  'diameter': np.float64(105.00863509052704)},
 {'timestep': 8640,
  'cell_count': 128,
  'diameter': np.float64(127.58915261933467)},
 {'timestep': 9360,
  'cell_count': 256,
  'diameter': np.f

In [20]:
df_summary = pd.DataFrame(results)
df_summary.to_csv(output_folder + "summary.csv", index=False)